# 🔍 08 XAI & Explainability Visuals
Go beyond predictions and understand the neural logic.

### 🎯 Goals:
1. Implement Grad-CAM (Gradient-weighted Class Activation Mapping) for MobileNetV2
2. Implement Saliency Maps
3. Save visualizations to `/outputs/xai/`

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive'): drive.mount('/content/drive')

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2

def get_gradcam_heatmap(model, img_array, last_conv_layer_name='out_relu'):
    # Handle nested models (e.g., Sequential with MobileNetV2 as a layer)
    target_layer = None
    for layer in model.layers:
        if hasattr(layer, 'get_layer'):
            try:
                target_layer = layer.get_layer(last_conv_layer_name)
                base_model = layer
                break
            except ValueError:
                continue
    
    if target_layer is None:
        # Try top level
        try:
            target_layer = model.get_layer(last_conv_layer_name)
            base_model = model
        except ValueError:
            raise ValueError(f"Layer {last_conv_layer_name} not found in model.")

    # Build a model that maps the input image to the activations of the last conv layer as well as the output predictions
    # This is tricky with nested models. A safer way is to build a new model from base_model inputs to target_layer output
    grad_model = tf.keras.models.Model(
        [model.inputs], [target_layer.output, model.output]
    )
    
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        class_channel = preds[:, tf.argmax(preds[0])]
    
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()